In [ ]:
!pip install -U albumentations segmentation-models-pytorch iterative-stratification


In [ ]:
#std library

import os
import math
import random
import time
from typing import Sequence, Tuple, List, Optional
from typing import Callable
from collections import Counter

#image and visualization

import cv2
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import clear_output

#data handling and preprocessing

import numpy as np
import torchvision.transforms.functional as TF
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

#pytorch core

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as BaseDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

#models

import segmentation_models_pytorch as smp

#utility

import itertools


In [ ]:
# On Google Colab the project folder lives on Google Drive; elsewhere this cell does nothing
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# Paths. Set the PROJECT_ROOT environment variable (or edit the default) to use a folder
# other than the Google Drive one; the layout inside it must stay the same.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "/content/drive/MyDrive/ML_Project_Work")
DATA_ROOT = os.path.join(PROJECT_ROOT, "train/archive/train")   # one sub-folder per sample: rgb.jpg + labels.png
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "modelstages")       # best checkpoint of each training phase
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

dataset_root = DATA_ROOT

# Seeds for Python, NumPy and PyTorch. Runs are repeatable on the same setup,
# but not bit-exact: some cuDNN kernels are non-deterministic.
SEED = 69
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

The `UNISADataset` class handles the loading, augmentation, and preprocessing of images and masks from a list of directories. It is designed to be used with the PyTorch `DataLoader`.


In [ ]:
class UNISADataset(BaseDataset):

    def __init__(self, sample_dirs, augmentation=None, preprocessing=None):
        self.sample_dirs = sample_dirs
        self.augmentation = augmentation
        self.preprocessing = preprocessing

    def __getitem__(self, idx):
        sample_path = self.sample_dirs[idx]

        # loading RGB image
        image = Image.open(os.path.join(sample_path, "rgb.jpg")).convert("RGB")
        image = np.array(image)

        # loading mask in palette mode (integer values from 0 to 8)
        mask = Image.open(os.path.join(sample_path, "labels.png")).convert("P")
        mask = np.array(mask)

        # apply augmentation
        if self.augmentation:
            augmented = self.augmentation(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        # apply preprocessing
        if self.preprocessing:
            processed = self.preprocessing(image=image, mask=mask)
            image, mask = processed["image"], processed["mask"]

        return image, mask.long()

    def __len__(self):
        return len(self.sample_dirs)




# Data augmentation and preprocessing pipeline

In [ ]:
def get_training_augmentation():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.ShiftScaleRotate(shift_limit=0.1, rotate_limit=20, scale_limit=0.2, p=0.5),
        A.ElasticTransform(alpha=50, sigma=5, p=0.3),
        A.RandomBrightnessContrast(p=0.8),
        A.ColorJitter(p=0.3),
        A.HueSaturationValue(p=0.3),
        A.GaussianBlur(p=0.2)
    ])



def get_preprocessing():
    return A.Compose([
        A.Resize(height=544, width=1024),       # avoiding inconsistencies for augmentation
        A.Normalize(mean=(0.485,0.456,0.406),       #imagenet mean and std
                    std=(0.229,0.224,0.225)),
     #   A.Normalize(mean=(0.437,0.456,0.432),       #raw dataset mean and std
     #               std=(0.266,0.265,0.267)),
        ToTensorV2()
    ])



# Utility
Function to visualize the original images and their corresponding augmentations to ensure they do not distort the image.


In [ ]:
# Function to visualize each transformation
def plot_augmentation_steps(image, transformations):
    fig, axs = plt.subplots(1, len(transformations) + 1, figsize=(22, 5))

    # Original image
    axs[0].imshow(image)
    axs[0].set_title("Original")
    axs[0].axis("off")

    # Apply each transformation individually
    for i, (name, transform) in enumerate(transformations):
        augmented = transform(image=image)['image']
        axs[i + 1].imshow(augmented)
        axs[i + 1].set_title(name)
        axs[i + 1].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# any sample works; 0022 is the one shown in the report
sample_id = '0022' if os.path.isdir(os.path.join(DATA_ROOT, '0022')) else sorted(os.listdir(DATA_ROOT))[0]
image_path = os.path.join(DATA_ROOT, sample_id, 'rgb.jpg')
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# List of transformations (one at a time)
transformations = [
    ("HorizontalFlip", A.HorizontalFlip(p=1.0)),
    ("VerticalFlip", A.VerticalFlip(p=1.0)),
    ("ShiftScaleRotate", A.ShiftScaleRotate(shift_limit=0.1, rotate_limit=20, scale_limit=0.2, p=1.0)),
    ("ElasticTransform", A.ElasticTransform(alpha=50, sigma=5, p=1.0)),
    ("RandomBrightnessContrast", A.RandomBrightnessContrast(p=1.0)),
    ("ColorJitter", A.ColorJitter(p=1.0)),
    ("HueSaturationValue", A.HueSaturationValue(p=1.0)),
    ("GaussianBlur", A.GaussianBlur(p=1.0)),
]

# Step-by-step visualization of the effect of transformations
plot_augmentation_steps(image, transformations)



# Dataset split

The `split_dataset_dirs_stratified_3way` function performs a multilabel stratified split of the dataset into train, validation, and test sets. The stratified split ensures that the distribution of classes (0–8) present in the masks is similarly represented across each subset. It uses `MultilabelStratifiedShuffleSplit` in two steps: first, it separates the test set from the combined train+val block, then it further splits this block into train and validation sets. This approach is essential in segmentation datasets where each image may contain multiple classes.



In [ ]:
NUM_CLASSES = 9

def split_dataset_dirs_stratified_3way(
    root_dir: str,
    test_ratio: float,
    val_ratio_train: float,
    random_seed: int
):

    # collect the IDs of all samples
    all_ids = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    # build the multilabel matrix (n_samples × NUM_CLASSES)
    multilabel_matrix = []
    for sample_id in all_ids:
        mask_path = os.path.join(root_dir, sample_id, "labels.png")
        mask = np.array(Image.open(mask_path).convert("P"))
        unique_classes = np.unique(mask)

        labels = np.zeros(NUM_CLASSES, dtype=int)
        labels[unique_classes] = 1
        multilabel_matrix.append(labels)

    multilabel_matrix = np.array(multilabel_matrix)

    # STAGE 1: train+val vs test
    msss = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=test_ratio, random_state=random_seed
    )
    train_val_idx, test_idx = next(msss.split(all_ids, multilabel_matrix))

    # temporary lists
    train_val_ids = [all_ids[i] for i in train_val_idx]
    test_ids      = [all_ids[i] for i in test_idx]

    # STAGE 2: train vs val
    msss2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=val_ratio_train, random_state=random_seed
    )
    tv_matrix = multilabel_matrix[train_val_idx]   # labels only for the train+val block
    train_idx, val_idx = next(msss2.split(train_val_ids, tv_matrix))

    train_ids = [train_val_ids[i] for i in train_idx]
    val_ids   = [train_val_ids[i] for i in val_idx]

    # convert to full paths
    train_dirs = [os.path.join(root_dir, d) for d in train_ids]
    val_dirs   = [os.path.join(root_dir, d) for d in val_ids]
    test_dirs  = [os.path.join(root_dir, d) for d in test_ids]

    return train_dirs, val_dirs, test_dirs, all_ids, train_ids, val_ids, test_ids


In [ ]:
# stratified split
train_dirs, val_dirs, test_dirs, all_ids, train_ids, val_ids, test_ids = split_dataset_dirs_stratified_3way(
    dataset_root,
    test_ratio=0.10,
    val_ratio_train=0.20,
    random_seed=69
)

# preprocessing, dataset, dataloader
preprocessing = get_preprocessing()
augmentation = get_training_augmentation()

train_dataset = UNISADataset(train_dirs, augmentation=augmentation, preprocessing=preprocessing)
val_dataset   = UNISADataset(val_dirs,   preprocessing=preprocessing)
test_dataset  = UNISADataset(test_dirs,  preprocessing=preprocessing)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=2)


# Utility 2
The `compute_dataset_mean_std` function calculates the mean and standard deviation per RGB channel across all images in the dataset to assess a custom normalization. Although ImageNet statistics are commonly used, comparing them with those from the rural domain was helpful to check compatibility. This comparison helps decide whether to keep the standard normalization or adapt it to improve performance.



In [ ]:
def compute_dataset_mean_std(root_dir, image_name="rgb.jpg"):

    sample_ids = [d for d in sorted(os.listdir(root_dir))
                  if os.path.isdir(os.path.join(root_dir, d))]

    channel_sums = np.zeros(3, dtype=np.float64)
    channel_sq_sums = np.zeros(3, dtype=np.float64)
    total_pixels = 0

    for sid in sample_ids:
        img_path = os.path.join(root_dir, sid, image_name)
        with Image.open(img_path).convert("RGB") as img:
            arr = np.array(img, dtype=np.float32) / 255.0  # shape (H, W, 3)
        pixels = arr.reshape(-1, 3)
        channel_sums += pixels.sum(axis=0)
        channel_sq_sums += (pixels ** 2).sum(axis=0)
        total_pixels += pixels.shape[0]

    mean = channel_sums / total_pixels
    var = channel_sq_sums / total_pixels - mean ** 2
    std = np.sqrt(var)
    return mean, std

In [ ]:

mean, std = compute_dataset_mean_std(dataset_root, image_name="rgb.jpg")

print("Dataset mean:", mean)
print("Dataset std :", std)


The `compute_class_distribution` function counts the number of pixels for each class in the masks, returning both the absolute and total class distribution. The `plot_class_distribution` function visualizes this distribution as percentages in a histogram to highlight any imbalances. This helps identify whether some classes are underrepresented.



In [ ]:
def compute_class_distribution(dataset_ids, base_dir):

    class_counts = Counter()

    for sample_id in dataset_ids:
        label_path = os.path.join(base_dir, sample_id, "labels.png")

        # Load the mask in 'P' mode to preserve the discrete class values
        label = Image.open(label_path).convert("P")
        label = np.array(label)

        # Count the pixels for each class
        unique, counts = np.unique(label, return_counts=True)
        class_counts.update(dict(zip(unique, counts)))

    total_pixels = sum(class_counts.values())
    return class_counts, total_pixels


CLASS_NAMES = [
    "background", "smooth trail", "traversable grass", "rough trail", "puddle",
    "obstacle", "non-traversable lw", "high vegetation", "sky"
]

def plot_class_distribution(class_counts, total_pixels, class_names=CLASS_NAMES):

    labels = sorted(class_counts.keys())
    counts = [class_counts[i] for i in labels]
    percentages = [count / total_pixels * 100 for count in counts]
    names = [class_names[i] for i in labels]

    plt.figure(figsize=(16, 6))
    plt.bar(names, percentages)
    plt.ylabel("Percentage of pixels (%)")
    plt.title("Class distribution in the dataset")
    plt.xticks(rotation=45, ha='center')
    plt.tight_layout()
    plt.grid(axis='y')
    plt.show()

The class distribution is computed and visualized for the train, validation, and test splits using pixel-wise counts from the segmentation masks. This helps identify class imbalance across subsets, which is critical in semantic segmentation tasks. Visualizing the distribution ensures that each split reflects a representative variety of classes, supporting robust training and evaluation.

In [ ]:
#train
class_counts_train, total_pixels_train = compute_class_distribution(train_ids, dataset_root)
plot_class_distribution(class_counts_train, total_pixels_train)

#validation
class_counts_val, total_pixels_val = compute_class_distribution(val_ids, dataset_root)
plot_class_distribution(class_counts_val, total_pixels_val)

#test
class_counts_val, total_pixels_val = compute_class_distribution(test_ids, dataset_root)
plot_class_distribution(class_counts_val, total_pixels_val)

# Training and validation functions





The `train_one_epoch` function performs a complete training epoch, updating the model's weights batch by batch and calculating the average loss.
The `validate` function evaluates the model on the validation set, calculating both the loss and the segmentation metrics.
In particular, it uses `compute_all_iou` and `compute_iou` to calculate the IoU for each class, ignoring the background.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    total_grad_norm = 0.0
    n_batches = 0

    for images, masks in dataloader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()

        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss



def validate(model, dataloader, loss_fn, device, num_classes=9):
    model.eval()
    total_loss = 0.0
    total_inter = np.zeros(num_classes, dtype=np.float64)
    total_union = np.zeros(num_classes, dtype=np.float64)

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            targets = masks.cpu().numpy()

            for pred, target in zip(preds, targets):
                iou_scores = compute_all_iou(target, pred, num_labels=num_classes - 1)
                # +1 because compute_all_iou skips 0 (background)
                for cls in range(1, num_classes):
                    iou = iou_scores[cls - 1]
                    if not np.isnan(iou):
                        union = np.sum((target == cls) | (pred == cls))
                        inter = iou * union
                        total_union[cls] += union
                        total_inter[cls] += inter

    per_cls_iou = np.divide(
        total_inter,
        total_union,
        out=np.full_like(total_inter, np.nan, dtype=np.float64),
        where=total_union != 0
    )

    valid = ~np.isnan(per_cls_iou)
    valid[0] = False

    miou = float(np.nanmean(per_cls_iou[valid]))
    avg_loss = total_loss / len(dataloader.dataset)

    return avg_loss, miou, per_cls_iou.tolist()


def compute_iou(mask1, mask2, label):
  intersection = np.sum((mask1 == label) & (mask2 == label))
  union = np.sum((mask1 == label) | (mask2 == label))
  if union == 0:
    return np.nan
  return intersection / union
def compute_all_iou(mask1, mask2, num_labels=8):
  iou_scores = np.zeros((num_labels))
  for label in range(num_labels):
    iou = compute_iou(mask1, mask2, label+1) # we skip the background label
    iou_scores[label] = iou
  return iou_scores



The `show_prediction_sample` function displays an input image, the ground-truth mask, and the model's prediction for a given index of the dataset. It performs denormalization of the image for proper visualization and applies the original palette to the masks to maintain color consistency.

In [ ]:
def show_prediction_sample(model, dataset, device, idx=0,
                           mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    model.eval()
    image, mask = dataset[idx]

    inp = image.unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(inp)
        pred = torch.argmax(out, dim=1).squeeze(0).cpu().numpy().astype('uint8')

    # Denormalization
    mean_t = torch.tensor(mean).view(3,1,1)
    std_t  = torch.tensor(std).view(3,1,1)
    img_denorm = image * std_t + mean_t
    img_np = img_denorm.permute(1,2,0).clamp(0,1).numpy()

    # Palette
    sample_path = dataset.sample_dirs[idx]
    pal = Image.open(os.path.join(sample_path, 'labels.png')).getpalette()

    gt_img = Image.fromarray(mask.numpy().astype('uint8'), mode='P')
    gt_img.putpalette(pal)
    gt_rgb = gt_img.convert('RGB')

    pred_img = Image.fromarray(pred, mode='P')
    pred_img.putpalette(pal)
    pred_rgb = pred_img.convert('RGB')

    fig, axs = plt.subplots(1, 3, figsize=(15,5))
    axs[0].imshow(img_np);    axs[0].set_title('Input')
    axs[1].imshow(gt_rgb);    axs[1].set_title('Ground Truth')
    axs[2].imshow(pred_rgb);  axs[2].set_title('Prediction')
    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


# UNet architecture with ResNet34 encoder

The model uses the U-Net architecture with a ResNet34 encoder pre-trained on ImageNet, leveraging general features already learned. This allows for effective transfer learning even on a small dataset, as in our case. The encoder is initially frozen, while only the last blocks of the decoder and the segmentation head are trained, reducing the number of updatable parameters and the risk of overfitting.

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    decoder_use_batchnorm=True,
    in_channels=3,
    classes=9
).to(device)

In [ ]:
# Choose device (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False


# Unfreeze the decoder and the head
for i in [-1, -2, -3, -4]:
    for param in model.decoder.blocks[i].parameters():
        param.requires_grad = True

for param in model.segmentation_head.parameters():
    param.requires_grad = True

model.to(device)

# 4. Summary of trainable parameters
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Model ready → trainable {train_params/1e6:.2f}M / "
    f"{total_params/1e6:.2f}M params"
)


In [ ]:
def print_trainable_layers(model):
    print("Layers with requires_grad=True:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"{name} | shape: {tuple(param.shape)}")

print_trainable_layers(model)

## Utility functions for training

In [ ]:
class WeightedFocalLoss(nn.Module):
    def __init__(self, weight: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(logits, target, weight=self.weight, reduction='none')
        probs = F.softmax(logits, dim=1)
        pt = probs.gather(1, target.unsqueeze(1)).squeeze(1)
        focal_term = (1 - pt) ** self.gamma
        return (focal_term * ce_loss).mean()

def get_class_weights(dataset_ids, base_dir, num_classes=9, max_w=20) -> torch.Tensor:
    counts, total = compute_class_distribution(dataset_ids, base_dir)
    freqs = {c: cnt / total for c, cnt in counts.items()}
    raw_weights = [1.0 / freqs.get(i, 1e-6) for i in range(num_classes)]
    raw_weights = [min(w, max_w) for w in raw_weights]
    weights = torch.tensor(raw_weights, dtype=torch.float32)
    return weights / weights.mean()

def get_loss_fn(name: str, class_weights: torch.Tensor, device="cuda") -> Callable:
    class_weights = class_weights.to(device)
    if name == "ce+dice":
        ce = nn.CrossEntropyLoss(weight=class_weights)
        dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
        return lambda pred, target: 0.5 * ce(pred, target) + 0.5 * dice(pred, target)
    elif name == "focal+dice":
        focal = smp.losses.FocalLoss(mode='multiclass', gamma=focal_gamma)
        dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
        return lambda pred, target: 0.7 * focal(pred, target) + 0.3 * dice(pred, target)
    elif name == "weighted_focal":
        return WeightedFocalLoss(weight=class_weights, gamma=focal_gamma)
    else:
        raise ValueError(f"Loss type '{name}' not recognized.")


def get_optimizer(model, lr, weight_decay):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

def get_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=plateau_factor,
        patience=plateau_patience,
        threshold=1e-3,
        threshold_mode='rel',
        cooldown=2,
        min_lr=plateau_min_lr
    )

The plot_training_curves function visualizes the training progress over epochs by generating three plots:

1. Training vs Validation Loss

2. Global mIoU (mean Intersection over Union)

3. Per-Class mIoU curves

It accepts sequences of loss and IoU values for each epoch and optionally adds a title suffix or saves the plots into a single PDF file. This visualization is useful for monitoring convergence, overfitting, and class-wise performance trends during training.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from typing import Sequence, Optional

def plot_training_curves(
    train_losses:     Sequence[float],
    val_losses:       Sequence[float],
    val_ious:         Sequence[float],
    val_class_ious:   Sequence[Sequence[float]],
    *,
    title_suffix: str = "",
    show: bool = True,
    save_path: Optional[str] = None
) -> None:

    # LOSS and global mIoU
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, linestyle='--', label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.plot(val_ious, label='Val mIoU', linestyle='-.')
    plt.title(f"Training/Validation Loss & mIoU{title_suffix}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss / mIoU")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # mIoU par class
    plt.figure(figsize=(10, 6))
    num_classes = len(val_class_ious[0])
    colors = plt.cm.tab10.colors
    # index 0 is the background: excluded from the IoU (always NaN), so not plotted
    for cls in range(1, num_classes):
        cls_curve = [epoch_iou[cls] for epoch_iou in val_class_ious]
        plt.plot(
            cls_curve,
            label=CLASS_NAMES[cls],
            color=colors[cls % len(colors)]
        )
    plt.title(f"Validation class mIoU{title_suffix}")
    plt.xlabel("Epoch")
    plt.ylabel("mIoU")
    plt.ylim(0, 1)
    plt.legend(ncol=2, fontsize="small")
    plt.grid(True)
    plt.tight_layout()

    # output
    if save_path:
        with PdfPages(save_path) as pdf:
            for fig_num in plt.get_fignums():
                pdf.savefig(plt.figure(fig_num))
        print(f"Grafichs saved in {save_path}")

    if show:
        plt.show()
    else:
        plt.close('all')


# Training process

# First stage

In [ ]:

def load_weights(model, checkpoint_path):

    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Weights loaded from {checkpoint_path}")


In [ ]:
# Optimizer
lr = 1e-3
weight_decay = 5e-5

# Scheduler ReduceLROnPlateau
plateau_factor = 0.5
plateau_patience = 3
plateau_min_lr = 1e-6

# Focal Loss
focal_gamma = 3.0
max_class_weight = 20.0

# Loss type and combinations
loss_type = "weighted_focal"  # options: "ce+dice", "focal+dice", "weighted_focal"

weights = get_class_weights(train_ids, dataset_root, max_w=max_class_weight)
loss_fn = get_loss_fn(loss_type, weights, device=device)
optimizer = get_optimizer(model, lr, weight_decay)
scheduler = get_scheduler(optimizer)

In [ ]:
best_iou = 0.0
lr_reduce_count = 0
n_epochs = 30
no_improve_epochs = 0
patience = 10



train_losses   = []
val_losses     = []
val_ious       = []
val_class_ious = []


for epoch in range(n_epochs):

    print(f"\nEpoch {epoch+1}/{n_epochs}")

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_iou, val_class_iou = validate(model, val_loader, loss_fn, device, num_classes=9)

    iou_str = " | ".join(f"{iou:.6f}" for iou in val_class_iou)
    print(f"IoU per classe: [{iou_str}]")

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ious.append(val_iou)
    val_class_ious.append(val_class_iou)

    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_iou)
    new_lr = optimizer.param_groups[0]['lr']

    if new_lr < old_lr:
        lr_reduce_count += 1
        print(f"  → Scheduler ha ridotto LR da {old_lr:.2e} a {new_lr:.2e} (decay count: {lr_reduce_count})")


    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")

    show_prediction_sample(model, val_dataset, device, idx=22)

    if val_iou > best_iou:
        best_iou = val_iou
        no_improve_epochs = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'FirstStageModel.pth'))
        print("Best model saved.")
    else:
        no_improve_epochs += 1
        print(f"No improvement in {no_improve_epochs} epochs.")

    # Early stopping
    if no_improve_epochs >= patience:
        print(f"Early stopping: no improvement in {patience} epochs.")
        break

plot_training_curves(
    train_losses,
    val_losses,
    val_ious,
    val_class_ious,
    title_suffix=" (resnet34)",
    save_path="training_curves_phase1.pdf"
)


# Second stage

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'FirstStageModel.pth')

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    decoder_use_batchnorm=True,
    in_channels=3,
    classes=9
).to(device)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)


for param in model.parameters():
    param.requires_grad = False


for i in [-1, -2, -3, -4]:
    for param in model.decoder.blocks[i].parameters():
        param.requires_grad = True

for param in model.segmentation_head.parameters():
    param.requires_grad = True

model.to(device)

load_weights(model, CHECKPOINT_PATH)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Model ready → trainable {train_params/1e6:.2f}M / "
    f"{total_params/1e6:.2f}M params"
)

In [ ]:
# Optimizer
lr = 3e-4
weight_decay = 5e-5

# Scheduler ReduceLROnPlateau
plateau_factor = 0.5
plateau_patience = 3
plateau_min_lr = 1e-6

# Focal Loss
focal_gamma = 3.0
max_class_weight = 20.0

# Loss type and combinations
loss_type = "ce+dice"  # options: "ce+dice", "focal+dice", "weighted_focal"

loss_fn = get_loss_fn(loss_type, weights, device=device)
optimizer = get_optimizer(model, lr, weight_decay)
scheduler = get_scheduler(optimizer)

In [ ]:
best_iou = 0.0
lr_reduce_count = 0
n_epochs = 50
no_improve_epochs = 0
patience = 10


train_losses   = []
val_losses     = []
val_ious       = []
val_class_ious = []


for epoch in range(n_epochs):

    print(f"\nEpoch {epoch+1}/{n_epochs}")

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_iou, val_class_iou = validate(model, val_loader, loss_fn, device, num_classes=9)

    iou_str = " | ".join(f"{iou:.6f}" for iou in val_class_iou)
    print(f"IoU per classe: [{iou_str}]")

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ious.append(val_iou)
    val_class_ious.append(val_class_iou)

    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_iou)
    new_lr = optimizer.param_groups[0]['lr']

    if new_lr < old_lr:
        lr_reduce_count += 1
        print(f"  → Scheduler ha ridotto LR da {old_lr:.2e} a {new_lr:.2e} (decay count: {lr_reduce_count})")


    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")

    show_prediction_sample(model, val_dataset, device, idx=22)

    if val_iou > best_iou:
        best_iou = val_iou
        no_improve_epochs = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'SecondStageModel.pth'))
        print("Best model saved.")
    else:
        no_improve_epochs += 1
        print(f"Nessun miglioramento da {no_improve_epochs} epoche.")

    # Early stopping
    if no_improve_epochs >= patience:
        print(f"Early stopping: nessun miglioramento per {patience} epoche.")
        break

plot_training_curves(
    train_losses,
    val_losses,
    val_ious,
    val_class_ious,
    title_suffix=" (resnet34)",
    save_path="training_curves_phase2.pdf"
)


# FINE TUNING

After the initial training with a frozen encoder, we reinitialize the optimizer with different learning rates for encoder, decoder, and head to enable targeted fine-tuning. We reset all training logs and gradient tracking to isolate the new training phase and avoid contaminating results. A learning rate scheduler is also reactivated to ensure adaptive learning control based on validation IoU, maintaining consistent optimization dynamics. This separation guarantees a clean and interpretable fine-tuning stage.

# First stage

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'SecondStageModel.pth')

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    decoder_use_batchnorm=True,
    in_channels=3,
    classes=9
).to(device)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)


for param in model.parameters():
    param.requires_grad = False


for i in [-1, -2, -3, -4]:
    for param in model.decoder.blocks[i].parameters():
        param.requires_grad = True

for param in model.segmentation_head.parameters():
    param.requires_grad = True

for name, module in model.encoder.named_children():
    if name in ['layer2', 'layer3', 'layer4']:
        for p in module.parameters():
            p.requires_grad = True

model.to(device)

load_weights(model, CHECKPOINT_PATH)

print_trainable_layers(model)


total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Model ready → trainable {train_params/1e6:.2f}M / "
    f"{total_params/1e6:.2f}M params"
)

In [ ]:
lr_enc = 1e-5
lr_dec = 1e-4
lr_head = 1e-4
wd = 1e-5

enc_params  = [p for p in model.encoder.parameters()           if p.requires_grad]
dec_params  = [p for p in model.decoder.parameters()           if p.requires_grad]
head_params = [p for p in model.segmentation_head.parameters() if p.requires_grad]


optimizer = torch.optim.AdamW([
    {"params": dec_params, "lr": lr_dec, "weight_decay": wd},
    {"params": enc_params, "lr": lr_enc, "weight_decay": wd},
    {"params": head_params, "lr": lr_head, "weight_decay": wd}
])


# Scheduler ReduceLROnPlateau
plateau_factor = 0.9
plateau_patience = 4
plateau_min_lr = 1e-7

# Focal Loss
focal_gamma = 3.0
max_class_weight = 20.0

# Loss type and combinations
loss_type = "weighted_focal"  # options: "ce+dice", "focal+dice", "weighted_focal"

loss_fn = get_loss_fn(loss_type, weights, device=device)
scheduler = get_scheduler(optimizer)

In [ ]:
best_iou          = 0.0
no_improve_epochs = 0
patience          = 7
n_epochs          = 30
lr_reduce_count   = 0

train_losses_fn   = []
val_losses_fn     = []
val_ious_fn       = []
val_class_ious_fn = []

for epoch in range(n_epochs):

    print(f"\nEpoch {epoch+1}/{n_epochs}")

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)

    val_loss, val_iou, val_class_iou = validate(model, val_loader, loss_fn, device, num_classes=9)

    iou_str = " | ".join(f"{iou:.6f}" for iou in val_class_iou)
    print(f"IoU per classe: [{iou_str}]")

    train_losses_fn.append(train_loss)
    val_losses_fn.append(val_loss)
    val_ious_fn.append(val_iou)
    val_class_ious_fn.append(val_class_iou)


    old_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_iou)
    new_lr = optimizer.param_groups[0]["lr"]

    if new_lr < old_lr:
        lr_reduce_count += 1
        print(f"  → Scheduler ha ridotto LR da {old_lr:.2e} a {new_lr:.2e}  "
              f"(decay count: {lr_reduce_count})")


    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val IoU: {val_iou:.4f}")

    show_prediction_sample(model, val_dataset, device, idx=22)

    if val_iou > best_iou:
        best_iou = val_iou
        no_improve_epochs = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'ThirdStageModel.pth'))
        print("Best model saved.")
    else:
        no_improve_epochs += 1
        print(f"Nessun miglioramento da {no_improve_epochs} epoche.")

    # early stopping
    if no_improve_epochs >= patience:
        print(f"Early stopping: nessun miglioramento per {patience} epoche.")
        break

print(f"\nFine-tuning terminato — best mIoU: {best_iou:.4f}")

plot_training_curves(
    train_losses_fn,
    val_losses_fn,
    val_ious_fn,
    val_class_ious_fn,
    title_suffix=" (resnet34)",
    save_path="training_curves_phase3.pdf"
)

# Second stage

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'ThirdStageModel.pth')


In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    decoder_use_batchnorm=True,
    in_channels=3,
    classes=9
).to(device)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)


for param in model.parameters():
    param.requires_grad = False


for i in [-1, -2, -3, -4]:
    for param in model.decoder.blocks[i].parameters():
        param.requires_grad = True

for param in model.segmentation_head.parameters():
    param.requires_grad = True

for name, module in model.encoder.named_children():
    if name in ['layer2', 'layer3', 'layer4']:
        for p in module.parameters():
            p.requires_grad = True

model.to(device)

load_weights(model, CHECKPOINT_PATH)

print_trainable_layers(model)


total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Model ready → trainable {train_params/1e6:.2f}M / "
    f"{total_params/1e6:.2f}M params"
)

In [ ]:
lr_enc = 1e-6
lr_dec = 1e-5
lr_head = 5e-5
wd = 1e-4

enc_params  = [p for p in model.encoder.parameters()           if p.requires_grad]
dec_params  = [p for p in model.decoder.parameters()           if p.requires_grad]
head_params = [p for p in model.segmentation_head.parameters() if p.requires_grad]


optimizer = torch.optim.AdamW([
    {"params": dec_params, "lr": lr_dec, "weight_decay": wd},
    {"params": enc_params, "lr": lr_enc, "weight_decay": wd},
    {"params": head_params, "lr": lr_head, "weight_decay": wd}
])


# Scheduler ReduceLROnPlateau
plateau_factor = 0.9
plateau_patience = 4
plateau_min_lr = 1e-7

# Focal Loss
focal_gamma = 3.0
max_class_weight = 20.0

# Loss type and combinations
loss_type = "ce+dice"  # options: "ce+dice", "focal+dice", "weighted_focal"

loss_fn = get_loss_fn(loss_type, weights, device=device)
scheduler = get_scheduler(optimizer)

In [ ]:
best_iou          = 0.0
no_improve_epochs = 0
patience          = 7
n_epochs          = 30
lr_reduce_count   = 0

train_losses_fn   = []
val_losses_fn     = []
val_ious_fn       = []
val_class_ious_fn = []

for epoch in range(n_epochs):

    print(f"\nEpoch {epoch+1}/{n_epochs}")

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)

    val_loss, val_iou, val_class_iou = validate(model, val_loader, loss_fn, device, num_classes=9)

    iou_str = " | ".join(f"{iou:.6f}" for iou in val_class_iou)
    print(f"IoU per classe: [{iou_str}]")

    train_losses_fn.append(train_loss)
    val_losses_fn.append(val_loss)
    val_ious_fn.append(val_iou)
    val_class_ious_fn.append(val_class_iou)


    old_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_iou)
    new_lr = optimizer.param_groups[0]["lr"]

    if new_lr < old_lr:
        lr_reduce_count += 1
        print(f"  → Scheduler ha ridotto LR da {old_lr:.2e} a {new_lr:.2e}  "
              f"(decay count: {lr_reduce_count})")


    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Learning Rate: {current_lr:.6f}")

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val IoU: {val_iou:.4f}")

    show_prediction_sample(model, val_dataset, device, idx=22)

    if val_iou > best_iou:
        best_iou = val_iou
        no_improve_epochs = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'FourthStageModel.pth'))
        print("Best model saved.")
    else:
        no_improve_epochs += 1
        print(f"Nessun miglioramento da {no_improve_epochs} epoche.")

    # early stopping
    if no_improve_epochs >= patience:
        print(f"Early stopping: nessun miglioramento per {patience} epoche.")
        break

print(f"\nFine-tuning terminato — best mIoU: {best_iou:.4f}")

plot_training_curves(
    train_losses_fn,
    val_losses_fn,
    val_ious_fn,
    val_class_ious_fn,
    title_suffix=" (resnet34)",   # opzionale
    save_path="training_curves_phase4.pdf"  # opzionale
)

# Testing


In [ ]:
test_loss, test_miou, test_per_cls = validate(
    model, test_loader, loss_fn, device, num_classes=NUM_CLASSES
)
print(f"Test  Loss: {test_loss:.4f} | Test mIoU: {test_miou:.4f}")


# Inference time

In [ ]:
import time

def measure_inference_time(model, dataloader, device: str = "cuda") -> float:

    model.eval()
    start_time = time.time()

    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(device, non_blocking=True)
            _ = model(images)

    end_time = time.time()
    elapsed = end_time - start_time
    return elapsed

print(f"Inference time: {measure_inference_time(model, test_loader, device):.2f}s")